# Step 2 · Competitive landscape of flying drones

Which applicants account for the most drone inventions, is inventive activity consolidating, who is entering, and which CPC subfields are growing? This notebook adapts sections 2, 3 and 5 of the professor’s competitive-landscape tutorial to the completed drone run. A DOCDB family is dated by its existing priority year. The applicant key is the entity ID published by the name-harmonization notebook.

On TIP, run the cells in order. Enter your existing run ID when prompted (for the completed production run, `phase2a-prod-001`). All results in this notebook are computed during execution. [Method, outputs and sharing instructions](README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import display

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "run_baseline.py").is_file())
sys.path.insert(0, str(ROOT))
from run_baseline import Run
from Step2.inputs import ATTRIBUTES, CPC, FAMILIES, FIRST_EVER, LINKS, load_first_ever, load_inputs
from Step2.competition import applicants_and_entry, concentration
from Step2.subfields import subfield_growth

if not os.environ.get("PATENT_RUN_ID"):
    os.environ["PATENT_RUN_ID"] = input("Existing PATENT_RUN_ID (for example phase2a-prod-001): ").strip()
run = Run.open(ROOT)
manifest = json.loads(run.verify("manifest.json").read_text(encoding="utf-8"))
if (manifest["env"], manifest["run_id"], manifest["config_sha256"]) != (run.env, run.run_id, run.config_sha256):
    raise ValueError("Selection manifest and run configuration differ.")
years = run.cfg["years"]
year_min, year_max = years["min"], years["max"]
last_complete = year_max - years["truncated"]
split_year, window, top_n = 2015, 3, 15
prefixes = manifest["methodology"]["cpc_prefixes"]
print(f"{run.env} / {run.run_id} | priority years {year_min}–{year_max} | complete through {last_complete}")
print("Sector CPC prefixes:", ", ".join(prefixes))

## 1. Cohort and applicant coverage

Families without an applicant stay in the extraction coverage count. The applicant metrics use families with an observed applicant link. Each joint family contributes one credit to each distinct linked entity. Multiple filings by that entity in the same family contribute a single credit. Step 0 review flags stay attached to the entities.

In [ ]:
families, links, family_cpc = load_inputs(run)
linked = links["unit_id"].nunique()
coverage = pd.DataFrame([{
    "selected_families": manifest["selected_families"],
    "extracted_families": len(families),
    "families_with_applicants": linked,
    "families_without_applicants": len(families) - linked,
    "family_applicant_credits": len(links),
    "applicant_entities": links["applicant"].nunique(),
    "entities_with_review_flags": links.loc[links["needs_manual_review"], "applicant"].nunique(),
    "families_with_any_cpc": family_cpc["unit_id"].nunique(),
}])
display(coverage.T.rename(columns={0: "count"}))

## 2. Concentration and consolidation

Shares divide each applicant’s family count by the sum of all applicant family credits. For complete years we report HHI (0–1 scale), equivalent number of applicants (1/HHI), CR4 and CR8 (%), sample coefficient of variation, Gini and the Lotka regression. These indicators measure inventive activity. Sales and market shares require separate evidence.

The tutorial specifies three-year windows. Here they start at the first configured year and do not overlap. The plot compares full three-year periods. The table also gives the final shorter period when applicable.

In [ ]:
overall, periods = concentration(links, year_min, last_complete, window=window)
display(overall.round(5))
display(periods[["period_start", "period_end", "full_window", "applicants", "hhi", "numbers_equivalent", "cr4_pct"]].round(5))

full = periods.loc[periods["full_window"]]
fig_concentration, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
labels = [f"{a}–{b}" for a, b in zip(full["period_start"], full["period_end"])]
for ax, column, title in zip(axes, ["hhi", "cr4_pct"], ["Herfindahl index (0–1 scale)", "Share of four largest applicants"]):
    ax.plot(labels, full[column], marker="o", color="#14665f")
    ax.set(title=title, xlabel="Priority-year window", ylim=(0, None))
    ax.tick_params(axis="x", labelrotation=45)
axes[1].yaxis.set_major_formatter(PercentFormatter(100))
plt.show()

## 3. Leading applicants and entry

The top-15 table covers the entire observed sector window, including recent incomplete years. An incumbent’s first observed drone family predates 2015. An entrant’s first observed drone family falls in 2015 or later. Founding dates or start-up status require external evidence. Use the review flag to inspect potential aliases among the leading applicants.

“New to patenting” requires a separate lookup across **all fields**. The first execution queries TIP in batches for applicant filings dated from 1900 through the configured end year. Reruns verify and reuse the saved lookup. Missing history remains visible. Each applicant’s country is the most frequent known country across distinct linked families, using alphabetical order to break ties.

In [ ]:
print("Loading earliest observed filing years across all fields …", flush=True)
first_ever = load_first_ever(run, links)
ranked, annual = applicants_and_entry(links, first_ever, year_min, year_max, last_complete, split_year)
leaders = ranked.head(top_n).copy()
print(f"Known first filing year for {first_ever.notna().sum():,} of {len(first_ever):,} entities")
display(leaders)
print(f"Among these {len(leaders)} applicants, {leaders['status'].eq('incumbent').sum()} are incumbents and "
      f"{leaders['status'].eq('entrant').sum()} are sector entrants.")

A family with multiple sector entrants counts once in the entrants’ share. Its denominator includes the year’s distinct families with applicant links. The disappearance rate uses an applicant’s last **observed** sector family and is reported only through three years before the last complete year. The beginning of the window can make established applicants look new to the sector. Shading marks incomplete recent years.

In [ ]:
display(annual.round(2))
fig_entry, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for key, label in [("active_applicants", "Active"), ("new_to_sector", "New to drones"), ("new_to_patenting", "New to patenting (known)")]:
    axes[0].plot(annual["year"], annual[key].to_numpy(dtype=float, na_value=float("nan")), label=label)
for key, label in [("entrant_family_share_pct", "Families with a sector entrant"), ("disappearance_pct", "Applicants last observed")]:
    axes[1].plot(annual["year"], annual[key], label=label)
axes[0].set(ylabel="Applicants", title="Applicant activity")
axes[1].set(ylabel="Share", title="Entry and disappearance", ylim=(0, 100))
axes[1].yaxis.set_major_formatter(PercentFormatter(100))
for ax in axes:
    ax.axvspan(last_complete + 0.5, year_max + 0.5, color="0.8", alpha=0.4)
    ax.set_xlabel("Family priority year")
    ax.legend(fontsize=8)
plt.show()

## 4. Growing CPC subfields

Match the sealed sector CPC prefixes, then group matching symbols by their first two digits after the slash. One family can belong to several subfields. Compare two successive five-year windows ending in the last complete year. A new group with zero families in the earlier window receives a blank growth rate. The three labels on the chart follow the tutorial’s median recent-size screen among groups with a positive baseline. Read small baselines alongside percentage growth.

In [ ]:
subfields, fastest = subfield_growth(family_cpc, prefixes, last_complete)
past_label = f"{last_complete - 9}–{last_complete - 5}"
recent_label = f"{last_complete - 4}–{last_complete}"
print(f"Earlier period {past_label}; recent period {recent_label}")
display(subfields.round(2))
print("Fastest growth among eligible subfields")
display(fastest.round(2))

comparable = subfields.dropna(subset=["growth_pct"])
fig_cpc, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
ax.scatter(comparable["recent_families"], comparable["growth_pct"], color="#14665f", alpha=0.65)
for name, row in fastest.iterrows():
    ax.annotate(name, (row["recent_families"], row["growth_pct"]), xytext=(4, 5), textcoords="offset points")
ax.axhline(0, color="0.5", linewidth=0.8)
ax.set(xlabel=f"Families in {recent_label}", ylabel=f"Growth over {past_label}", title="Flying-drone CPC subfields")
ax.yaxis.set_major_formatter(PercentFormatter(100))
plt.show()

## 5. PEST evidence

This evidence table concerns the European civil-drone market. Read each finding in its stated geography and period. The implications are interpretations for discussion. Open the named sources before quoting them in the deck.

In [ ]:
from hashlib import sha256
from io import BytesIO

pest_bytes = (ROOT / "Step2" / "pest.csv").read_bytes()
pest = pd.read_csv(BytesIO(pest_bytes))
with pd.option_context("display.max_colwidth", None):
    display(pest)

## 6. Publish the Step 2 tables and figures

The run publisher records input fingerprints and protects existing artifacts. The full applicant ranking stays in a CSV on TIP. The top 15, annual series, growth map and PEST table appear directly in the saved notebook for your teammates.

In [ ]:
app_inputs = (FAMILIES, LINKS, ATTRIBUTES)
growth_inputs = ("manifest.json", FAMILIES, CPC)
entry_inputs = (*app_inputs, FIRST_EVER)
settings = {"split_year": split_year, "window_years": window, "last_complete_year": last_complete}
growth_settings = {"prefixes": prefixes, "past_period": past_label, "recent_period": recent_label}

def publish_table(name, frame, inputs, metadata=None):
    path = run.publish(f"step2/{name}.csv", lambda out: frame.to_csv(out, index=False),
                       inputs=inputs, metadata=metadata)
    print(path.name, flush=True)

publish_table("coverage", coverage, ("manifest.json", *app_inputs, CPC))
publish_table("concentration", overall, app_inputs, settings)
publish_table("concentration_windows", periods, app_inputs, settings)
publish_table("applicant_ranking", ranked, entry_inputs, settings)
publish_table("top_applicants", leaders, ("step2/applicant_ranking.csv",), {"top_n": top_n})
publish_table("entry_exit", annual, entry_inputs, settings)
publish_table("cpc_subfields", subfields.reset_index(), growth_inputs, growth_settings)
publish_table("fastest_subfields", fastest.reset_index(), ("step2/cpc_subfields.csv",))
publish_table("pest", pest, (), {"source_file": "Step2/pest.csv", "sha256": sha256(pest_bytes).hexdigest()})

for name, figure, table in [
    ("concentration_trend", fig_concentration, "concentration_windows"),
    ("entry_exit", fig_entry, "entry_exit"),
    ("subfield_growth", fig_cpc, "cpc_subfields"),
]:
    run.publish(f"step2/{name}.png", lambda out: figure.savefig(out, dpi=150, format="png"),
                inputs=(f"step2/{table}.csv",))
print("Saved Step 2 outputs in", run.directory / "step2")

The competitive-landscape tutorial is by Grazia Sveva Ascione, Politecnico di Torino, drawing on original European Patent Office training material. This course-project adaptation is for non-commercial teaching and learning. The EPO has not authorised the adaptation and accepts no responsibility for its correctness. [Professor’s Step 2 notebook](https://github.com/Soroushkhayami/innovation-management-datascience26/blob/f1bca0b6ab2310d174cd9588c13f144b0e6492f1/tutorial_1/11_competitive_landscape.ipynb) · [Course terms](https://github.com/Soroushkhayami/innovation-management-datascience26/blob/f1bca0b6ab2310d174cd9588c13f144b0e6492f1/LICENSE.md)